# 🎙️ voicelegacy — Notebook de producción

**Clonación de voz por zero-shot con XTTS-v2, optimizado para Google Colab Free Tier (T4 GPU).**

Pipeline integrado con [speakerscribe](https://github.com/EnriqueForero/speakerscribe): consume sus JSON diarizados, construye un corpus curado de la voz objetivo, y sintetiza nuevo audio.

Powered by [coqui-tts (fork de Idiap)](https://github.com/idiap/coqui-ai-TTS) + XTTS-v2.

---

## ✅ Antes de empezar — checklist obligatorio

1. **Activa la GPU**: `Runtime → Change runtime type → T4 GPU`. Sin GPU, la inferencia es ~30× más lenta.

2. **Acepta la licencia CPML del modelo** antes de tocar nada: https://coqui.ai/cpml. Es uso no comercial / personal. Para legado familiar está dentro del scope; para una app comercial **léela completa**.

3. **Procesa primero tus entrevistas con `speakerscribe`**. Este notebook *consume* sus salidas, no transcribe. Si no tienes los `.json` diarizados, no puede continuar.

4. **Layout esperado del workspace en Drive**:
   ```
   <WORKSPACE>/
   ├── interviews_raw/        ← audio original (mp3/wav/m4a)
   ├── speakerscribe_out/     ← .json producidos por speakerscribe
   ├── reference_corpus/      ← (lo crea este notebook)
   ├── synthesis_out/         ← (lo crea este notebook)
   └── reports/               ← (lo crea este notebook)
   ```

5. **Reglas duras de calidad** (se aplican automáticamente):
   - Audio de llamadas (≤16 kHz) → **rechazado**. La compresión de codec contamina el clon.
   - Segmentos < 4 s o > 15 s → descartados.
   - SNR < 15 dB → descartado.
   - Solo se usan los top-N segmentos por score (SNR × duración).


---

## 1️⃣ Montar Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---

## 2️⃣ Configuración del run

**Esta es la única celda que debes editar.** Todo lo demás se autoconfigura.


In [2]:
# ══════════════════════════════════════════════════════════════════
#  ✏️  EDITAR AQUÍ — única celda que requiere cambios
# ══════════════════════════════════════════════════════════════════

# ── Workspace (carpeta raíz en Drive) ─────────────────────────────
WORKSPACE = '/content/drive/MyDrive/Pruebas/voicelegacy'

# ── Ruta a la librería voicelegacy en Drive (modo desarrollo) ────
# Sube la carpeta voicelegacy (con su pyproject.toml) a Drive y apunta aquí.
# Si vas a usar PyPI (futuro), deja esto en None y descomenta la celda B.
PACKAGE_DIR_IN_DRIVE = '/content/drive/MyDrive/Pruebas/voicelegacy'

# ── Speaker objetivo en los JSON de speakerscribe ────────────────
# Abre uno de los .transcript.md de speakerscribe y verifica qué etiqueta
# corresponde a la abuela. Suele ser 'SPEAKER_00' o 'SPEAKER_01'.
TARGET_SPEAKER = 'SPEAKER_01'

# ── Selección de corpus ──────────────────────────────────────────
TOP_N_SEGMENTS = 10            # Top N segmentos limpios para usar como referencia
MIN_SEGMENT_DURATION_S = 4.0   # Descartar segmentos más cortos
MAX_SEGMENT_DURATION_S = 15.0  # Descartar segmentos más largos
MIN_SNR_DB = 3.0              # SNR mínimo aceptable es 15
APPLY_DENOISE = True           # Reducción de ruido espectral

# ── Síntesis ──────────────────────────────────────────────────────
LANGUAGE = 'es'                # 'es', 'en', 'pt', 'it', 'fr', ...
TEMPERATURE = 0.7              # 0.5-0.9 — estabilidad vs expresividad
SPEED = 1.0                    # 0.8 = más lento, 1.2 = más rápido

# ── Textos a sintetizar (lista) ──────────────────────────────────
TEXTS_TO_SYNTHESIZE = [
    'Mi querido nieto, hoy quiero contarte algo importante: '
    'la paciencia es la virtud que más vale la pena cultivar.',
    'Cuando tengas dudas, escucha a las personas que te aman, '
    'pero al final decide tú, con la cabeza fría.',
]

# ── Aceptación de licencia del modelo (OBLIGATORIO) ──────────────
# Al poner True confirmas que leíste https://coqui.ai/cpml
ACCEPT_CPML = True  # ← cambia a True después de leer la licencia

# ── Re-procesar (overrides de caché) ─────────────────────────────
FORCE_REBUILD_REFERENCE = True  # True = reconstruir corpus aunque exista
FORCE_RESYNTHESIZE = False       # True = re-sintetizar aunque exista cache

# ── Smoke test ───────────────────────────────────────────────────
RUN_SMOKE_TEST = True            # Probar carga del modelo antes del run real


---

## 3️⃣ Pre-flight: verificar entorno (GPU, RAM, versiones)


In [3]:
import sys, platform, subprocess
from pathlib import Path

# Asegurar que el workspace existe
ws_root = Path(WORKSPACE)
if not ws_root.exists():
    print(f'⚠️  El workspace {WORKSPACE} no existe. Lo crearé.')
    ws_root.mkdir(parents=True, exist_ok=True)

# Python
print(f'🐍 Python {platform.python_version()}')
if sys.version_info >= (3, 13):
    print('   ⚠️  Python 3.13+ puede tener incompatibilidades con coqui-tts.')

# GPU
try:
    nvidia_out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        text=True,
    )
    print(f'🎮 GPU detectada: {nvidia_out.strip()}')
except (subprocess.CalledProcessError, FileNotFoundError):
    print('❌ No se detectó GPU. Ve a Runtime → Change runtime type → T4 GPU.')
    raise SystemExit('GPU requerida para velocidad razonable.')

# RAM
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f'💾 RAM total: {ram_gb:.1f} GB')
    if ram_gb < 11:
        print('   ⚠️  Menos de 11 GB. Posible OOM con archivos largos.')
except ImportError:
    pass

# CPML
if not ACCEPT_CPML:
    print()
    print('⛔ ACCEPT_CPML está en False.')
    print('   Lee https://coqui.ai/cpml — si aceptas, cambia ACCEPT_CPML=True arriba.')
    raise SystemExit('Aceptación de licencia pendiente.')

print('\n✅ Pre-flight OK')


🐍 Python 3.12.13
🎮 GPU detectada: Tesla T4, 15360 MiB
💾 RAM total: 12.7 GB

✅ Pre-flight OK


---

## 4️⃣ Instalar `voicelegacy` y dependencias

**OPCIÓN A — desarrollo**: instala desde Drive en modo editable.
**OPCIÓN B — producción**: instala desde PyPI (cuando esté publicado).

Después de instalar, **reinicia el runtime** (`Runtime → Restart runtime`) la primera vez.


In [4]:
%%time
# ── Opción A: instalar desde Drive (DEVELOPMENT) ────────────────
import subprocess, sys
from pathlib import Path

if PACKAGE_DIR_IN_DRIVE and (Path(PACKAGE_DIR_IN_DRIVE) / 'pyproject.toml').exists():
    print(f'📦 Instalando desde {PACKAGE_DIR_IN_DRIVE} (editable)...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', PACKAGE_DIR_IN_DRIVE, '--quiet'],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print('✅ voicelegacy instalado en modo editable')
        print('⚠️  Si es la PRIMERA instalación: Runtime → Restart runtime,')
        print('   luego re-corre desde la celda 1.')
    else:
        print('❌ Falló la instalación local:')
        print(result.stderr[-1500:])
        print('\n   Intenta la Opción B (PyPI) en la siguiente celda.')
else:
    print(f'⚠️  pyproject.toml no encontrado en {PACKAGE_DIR_IN_DRIVE}')
    print('   1) Sube la carpeta voicelegacy a Drive en esa ruta, O')
    print('   2) Usa la siguiente celda (PyPI), O')
    print('   3) Cambia PACKAGE_DIR_IN_DRIVE en la celda 2.')


📦 Instalando desde /content/drive/MyDrive/Pruebas/voicelegacy (editable)...
✅ voicelegacy instalado en modo editable
⚠️  Si es la PRIMERA instalación: Runtime → Restart runtime,
   luego re-corre desde la celda 1.
CPU times: user 3.08 ms, sys: 178 µs, total: 3.25 ms
Wall time: 19.5 s


In [5]:
# ── Opción B: instalar desde PyPI (PRODUCCIÓN) ──────────────────
# Descomenta SI no tienes el paquete en Drive. Después: Runtime → Restart runtime.

# %pip install -U voicelegacy --quiet
# print('✅ voicelegacy instalado/actualizado desde PyPI. Reinicia el runtime ahora.')


---

## 5️⃣ Validar import y versiones


In [3]:
%%time
# Wall time: 19.7 s
import voicelegacy
from voicelegacy import (
    PipelineConfig, ReferenceConfig, SynthesisConfig, WorkspacePaths,
    run_reference_phase, run_synthesis, run_batch_synthesis,
    configure_logging, release_model,
)

configure_logging(level='INFO')
print(f'✅ voicelegacy v{voicelegacy.__version__}')

# Validar coqui-tts
try:
    import TTS
    print(f'✅ coqui-tts importado (TTS v{getattr(TTS, "__version__", "?")})')
except ImportError:
    print('❌ coqui-tts NO está disponible. Re-corre la celda 4 e intenta de nuevo.')
    raise

# Validar torch + GPU
import torch
print(f'✅ PyTorch {torch.__version__}, CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   Device: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

16:09:29 | INFO     | voicelegacy.logging_config:configure_logging:67 | Logging configured (level=INFO, file=None)


✅ voicelegacy v0.1.0
✅ coqui-tts importado (TTS v0.27.5)
✅ PyTorch 2.10.0+cu128, CUDA disponible: True
   Device: Tesla T4
   VRAM  : 14.6 GB
CPU times: user 12.8 s, sys: 1.86 s, total: 14.6 s
Wall time: 16.1 s


---

## 6️⃣ Armar el `PipelineConfig` y crear directorios


In [4]:
paths = WorkspacePaths(workspace=WORKSPACE)
paths.mkdirs()

config = PipelineConfig(
    reference=ReferenceConfig(
        target_speaker_label=TARGET_SPEAKER,
        top_n_segments=TOP_N_SEGMENTS,
        min_segment_duration_s=MIN_SEGMENT_DURATION_S,
        max_segment_duration_s=MAX_SEGMENT_DURATION_S,
        min_snr_db=MIN_SNR_DB,
        apply_denoise=APPLY_DENOISE,
    ),
    synthesis=SynthesisConfig(
        language=LANGUAGE,
        temperature=TEMPERATURE,
        speed=SPEED,
    ),
    accept_coqui_tos=ACCEPT_CPML,
    force_rebuild_reference=FORCE_REBUILD_REFERENCE,
    force_resynthesize=FORCE_RESYNTHESIZE,
)

print(f'📁 Workspace      : {paths.workspace}')
print(f'   Entrevistas    : {paths.interviews_raw}')
print(f'   speakerscribe  : {paths.speakerscribe_out}')
print(f'   ref_corpus     : {paths.reference_corpus}')
print(f'   synth_out      : {paths.synthesis_out}')
print(f'   reports        : {paths.reports}')
print(f'   runs.db        : {paths.db_path}')
print()
print(f'🎯 Speaker objetivo: {config.reference.target_speaker_label}')
print(f'🔢 Top N           : {config.reference.top_n_segments}')
print(f'🌐 Idioma          : {config.synthesis.language}')

# Sanity: chequear que haya JSONs de speakerscribe
json_files = list(paths.speakerscribe_out.glob('*.json'))
if not json_files:
    print()
    print(f'⚠️  No hay archivos .json en {paths.speakerscribe_out}.')
    print('   Procesa las entrevistas con speakerscribe y copia los .json ahí.')
else:
    print(f'\n✅ {len(json_files)} archivo(s) .json encontrados:')
    for jf in json_files[:5]:
        print(f'   • {jf.name}')
    if len(json_files) > 5:
        print(f'   ... y {len(json_files) - 5} más')


📁 Workspace      : /content/drive/MyDrive/Pruebas/voicelegacy
   Entrevistas    : /content/drive/MyDrive/Pruebas/voicelegacy/interviews_raw
   speakerscribe  : /content/drive/MyDrive/Pruebas/voicelegacy/speakerscribe_out
   ref_corpus     : /content/drive/MyDrive/Pruebas/voicelegacy/reference_corpus
   synth_out      : /content/drive/MyDrive/Pruebas/voicelegacy/synthesis_out
   reports        : /content/drive/MyDrive/Pruebas/voicelegacy/reports
   runs.db        : /content/drive/MyDrive/Pruebas/voicelegacy/runs.db

🎯 Speaker objetivo: SPEAKER_01
🔢 Top N           : 10
🌐 Idioma          : es

✅ 1 archivo(s) .json encontrados:
   • 2026-05-15 _Cita oftalmológica_large-v3.json


---

## 7️⃣ Smoke test — verificar que XTTS-v2 carga y genera

Antes del run real, sintetiza una frase corta con un archivo de referencia mínimo. Confirma que:
- coqui-tts descarga los pesos (la primera vez baja ~2 GB).
- La GPU funciona.
- La librería completa el ciclo sin errores.

Si esto falla, **no pases a la fase 1**.


In [6]:
%%time
# Wall time: 48.1 s
import numpy as np, soundfile as sf
from voicelegacy.synthesis import load_xtts_model, synthesize_to_file

if RUN_SMOKE_TEST:
    # Audio sintético de 8s — solo para probar el pipeline mecánicamente.
    smoke_dir = paths.reports / '_smoke'
    smoke_dir.mkdir(exist_ok=True)
    smoke_ref = smoke_dir / 'fake_ref.wav'

    sr = 22050
    t = np.arange(int(sr * 8)) / sr
    y = 0.30 * np.sin(2 * np.pi * 220 * t) + 0.15 * np.sin(2 * np.pi * 440 * t)
    y *= 0.5 + 0.5 * np.sin(2 * np.pi * 3 * t)  # envelope
    sf.write(str(smoke_ref), y.astype(np.float32), sr)

    print('⏬ Cargando XTTS-v2 (primera vez descarga ~2 GB)...')
    tts = load_xtts_model(config.synthesis, accept_tos=config.accept_coqui_tos)

    smoke_out = smoke_dir / 'smoke_test.wav'
    synthesize_to_file(
        tts=tts,
        text='Esto es una prueba de funcionamiento del pipeline.',
        speaker_wav=smoke_ref,
        output_path=smoke_out,
        config=config.synthesis,
    )
    print(f'\n✅ Smoke test OK — output: {smoke_out}')
    print('   (Sonará raro: la referencia es sintética. Era solo prueba mecánica.)')
else:
    print('⏭️  Smoke test omitido (RUN_SMOKE_TEST=False)')

15:56:25 | INFO     | voicelegacy.synthesis:load_xtts_model:78 | Loading XTTS model 'tts_models/multilingual/multi-dataset/xtts_v2' on device 'cuda'


⏬ Cargando XTTS-v2 (primera vez descarga ~2 GB)...


100%|██████████| 1.87G/1.87G [00:23<00:00, 79.4MiB/s]
4.37kiB [00:00, 9.65MiB/s]
361kiB [00:00, 58.7MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 84.6kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 52.9MiB/s]
15:57:06 | INFO     | voicelegacy.synthesis:load_xtts_model:84 | Model loaded and cached.
15:57:06 | INFO     | voicelegacy.synthesis:synthesize_to_file:149 | Synthesizing 50 char(s) to smoke_test.wav with 1 ref file(s)
15:57:13 | INFO     | voicelegacy.synthesis:synthesize_to_file:171 | Wrote /content/drive/MyDrive/Pruebas/voicelegacy/reports/_smoke/smoke_test.wav (24000 Hz)



✅ Smoke test OK — output: /content/drive/MyDrive/Pruebas/voicelegacy/reports/_smoke/smoke_test.wav
   (Sonará raro: la referencia es sintética. Era solo prueba mecánica.)
CPU times: user 34.4 s, sys: 7.22 s, total: 41.6 s
Wall time: 48.1 s


---

## 8️⃣ Fase 1: Construir el corpus de referencia

Lee los `.json` de `speakerscribe_out/`, extrae los segmentos del speaker objetivo, los limpia (denoise → trim → normalize), los scorea y guarda los mejores en `reference_corpus/`.

Idempotente: si ya hay WAVs en `reference_corpus/`, los reusa (salvo `FORCE_REBUILD_REFERENCE=True`).


In [5]:
# ── Convertir audio/video a WAV para que corpus.py lo encuentre ──
# Convierte todos los archivos no-WAV de interviews_raw/ a WAV 22050 Hz mono.
# Omite archivos que ya tienen su WAV correspondiente.
# No requiere indicar nombres — procesa todo lo que encuentre.

import subprocess
from pathlib import Path

# Extensiones que ffmpeg puede convertir a WAV
CONVERTIBLES = {'.mp4', '.mp3', '.m4a', '.mkv', '.ogg', '.flac', '.aac', '.mov', '.webm'}

raw_dir = paths.interviews_raw
candidatos = [f for f in sorted(raw_dir.iterdir())
              if f.suffix.lower() in CONVERTIBLES]

if not candidatos:
    print(f'ℹ️  No hay archivos para convertir en {raw_dir}')
    print('   Archivos presentes:')
    for f in sorted(raw_dir.iterdir()):
        print(f'     • {f.name}')
else:
    print(f'📂 {len(candidatos)} archivo(s) convertible(s) encontrado(s)\n')
    for src in candidatos:
        dst = src.with_suffix('.wav')
        if dst.exists():
            print(f'♻️  Ya existe: {dst.name} — omitido')
            continue
        print(f'🔄 {src.name} → {dst.name}')
        result = subprocess.run(
            ['ffmpeg', '-i', str(src),
             '-ar', '22050',  # sample rate que espera XTTS-v2
             '-ac', '1',      # mono
             '-y',            # sobreescribir si existe
             str(dst)],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            print(f'   ✅ {dst.stat().st_size / 1e6:.1f} MB')
        else:
            print(f'   ❌ ffmpeg falló:\n{result.stderr[-600:]}')

print('\n✅ Conversión completada')

ℹ️  No hay archivos para convertir en /content/drive/MyDrive/Pruebas/voicelegacy/interviews_raw
   Archivos presentes:
     • 2026-05-15 _Cita oftalmológica.wav

✅ Conversión completada


In [ ]:
# ── Corregir clipping en reference_corpus/ ───────────────────────
# Los segmentos con peak >= -1 dBFS están saturados.
# Este paso los normaliza a -3 dBFS sin tocar los que ya están bien.

import numpy as np
import soundfile as sf
from pathlib import Path

corpus_dir = paths.reference_corpus
wavs = sorted(corpus_dir.glob('*.wav'))
print(f'📂 {len(wavs)} WAVs en reference_corpus/\n')

PEAK_TARGET = 0.708   # -3 dBFS  (10^(-3/20))
CLIP_THRESHOLD = 0.99 # considera clipping si peak > esto

corregidos = 0
for wav_path in wavs:
    y, sr = sf.read(str(wav_path), dtype='float32')
    peak = float(np.max(np.abs(y)))

    if peak > CLIP_THRESHOLD:
        y_norm = y * (PEAK_TARGET / peak)
        sf.write(str(wav_path), y_norm, sr)
        print(f'  🔧 {wav_path.name}  peak {peak:.3f} → {PEAK_TARGET:.3f}')
        corregidos += 1
    else:
        print(f'  ✅ {wav_path.name}  peak {peak:.3f} — sin cambio')

print(f'\n✅ {corregidos} archivo(s) corregidos')

In [6]:
import json
from pathlib import Path

json_file = list(paths.speakerscribe_out.glob('*.json'))[0]
data = json.loads(json_file.read_text(encoding='utf-8'))

print(f'JSON: {json_file.name}')
print(f'source_audio en JSON: {repr(data.get("source_audio") or data.get("audio_file"))}')
print()
print('Archivos en interviews_raw/:')
for f in sorted(paths.interviews_raw.iterdir()):
    print(f'  • {repr(f.name)}')

JSON: 2026-05-15 _Cita oftalmológica_large-v3.json
source_audio en JSON: '2026-05-15 _Cita oftalmológica.wav'

Archivos en interviews_raw/:
  • '2026-05-15 _Cita oftalmológica.wav'


In [ ]:
# # ── Denoising previo al pipeline ──────────────────────────────────
# # Aplica reducción de ruido agresiva al archivo completo.
# # Si el SNR sube a >8 dB en algunos segmentos, el pipeline puede continuar
# # con MIN_SNR_DB relajado. Si sigue en 3-4 dB, el audio no es recuperable.

# import subprocess, sys
# from pathlib import Path

# # Instalar noisereduce y dependencias si no están
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
#                 'noisereduce', 'librosa', 'soundfile'], check=False)

# import numpy as np
# import librosa
# import noisereduce as nr
# import soundfile as sf

# raw_dir = paths.interviews_raw
# wav_in  = raw_dir / '2026-05-14 _Job Rotation.wav'   # el que convertiste antes
# wav_out = raw_dir / '2026-05-14 _Job Rotation_denoised.wav'

# if not wav_in.exists():
#     print(f'❌ No se encuentra {wav_in.name}')
#     print('   Archivos disponibles:')
#     for f in raw_dir.iterdir():
#         print(f'   • {f.name}')
# else:
#     print(f'📂 Cargando {wav_in.name}...')
#     y, sr = librosa.load(str(wav_in), sr=22050, mono=True)
#     print(f'   Duración: {len(y)/sr:.1f}s  |  SR: {sr} Hz')

#     print('🔇 Aplicando denoising agresivo (prop_decrease=0.95)...')
#     # Usar los primeros 2s como muestra de ruido (suele ser silencio/ruido de fondo)
#     noise_sample = y[:sr*2]
#     y_clean = nr.reduce_noise(
#         y=y, sr=sr,
#         y_noise=noise_sample,
#         prop_decrease=0.95,
#         stationary=False,    # ruido no estacionario (más agresivo)
#     )

#     print(f'💾 Guardando {wav_out.name}...')
#     sf.write(str(wav_out), y_clean.astype(np.float32), sr)
#     size_mb = wav_out.stat().st_size / 1e6
#     print(f'✅ Guardado ({size_mb:.1f} MB)')
#     print()
#     print('Siguiente paso:')
#     print('  1. En el JSON de speakerscribe, el campo source_audio apunta al nombre')
#     print('     sin extensión. Necesitas que el archivo denoised tenga el mismo nombre.')
#     print('  2. Renombra (o copia) el denoised para que coincida:')
#     print(f'     wav_out.rename(raw_dir / "2026-05-14 _Job Rotation_denoised_22k.wav")')
#     print('  3. Luego modifica el JSON para que source_audio apunte al nuevo nombre,')
#     print('     O usa la celda de parche más abajo.')

📂 Cargando 2026-05-14 _Job Rotation.wav...
   Duración: 1626.3s  |  SR: 22050 Hz
🔇 Aplicando denoising agresivo (prop_decrease=0.95)...
💾 Guardando 2026-05-14 _Job Rotation_denoised.wav...
✅ Guardado (71.7 MB)

Siguiente paso:
  1. En el JSON de speakerscribe, el campo source_audio apunta al nombre
     sin extensión. Necesitas que el archivo denoised tenga el mismo nombre.
  2. Renombra (o copia) el denoised para que coincida:
     wav_out.rename(raw_dir / "2026-05-14 _Job Rotation_denoised_22k.wav")
  3. Luego modifica el JSON para que source_audio apunte al nuevo nombre,
     O usa la celda de parche más abajo.


In [7]:
%%time
corpus_result = run_reference_phase(paths, config)

print()
print('📊 Resumen fase 1:')
print(f'   Candidatos totales : {len(corpus_result.all_wavs)}')
print(f'   Aprobados (pass)   : {sum(1 for r in corpus_result.reports if r.passed)}')
print(f'   Top seleccionados  : {len(corpus_result.top_wavs)}')

if corpus_result.top_wavs:
    print()
    print('🏆 Top segmentos por score:')
    top_reports = sorted(
        [r for r in corpus_result.reports if r.passed],
        key=lambda r: r.score,
        reverse=True,
    )
    for r in top_reports[:TOP_N_SEGMENTS]:
        print(f'   • score={r.score:.3f}  dur={r.stats.duration_s:5.2f}s  '
              f'snr={r.stats.snr_db:5.1f}dB  {r.path.name}')
else:
    print()
    print('❌ No se construyó corpus. Causas más probables:')
    print('   1. ¿Hay .json en speakerscribe_out/? (re-corre la celda 6)')
    print('   2. ¿El TARGET_SPEAKER coincide con la etiqueta real en los JSON?')
    print('      Abre uno y verifica.')
    print('   3. ¿El audio fuente es de llamada (≤16 kHz)? Está rechazado por diseño.')
    print('   4. ¿Las grabaciones son muy cortas o muy ruidosas? Relaja MIN_SNR_DB o')
    print('      MIN_SEGMENT_DURATION_S en la celda 2 — entendiendo el costo de calidad.')
    raise SystemExit('Fase 1 sin output. No continúo a fase 2.')

16:09:51 | WARNING  | voicelegacy.pipeline:run_reference_phase:84 | force_rebuild_reference=True — wiping reference_corpus/
16:09:51 | INFO     | voicelegacy.corpus:load_speakerscribe_json:115 | Parsed 219 segments from 2026-05-15 _Cita oftalmológica_large-v3.json
16:09:51 | INFO     | voicelegacy.corpus:filter_segments:143 | Kept 24/219 segments for speaker 'SPEAKER_01' (duration 4.0-15.0s)
16:09:51 | INFO     | voicelegacy.corpus:extract_segments_to_wav:187 | Loaded source: 2026-05-15 _Cita oftalmológica.wav (14,593,220 samples)
/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
/usr/local/lib/python3.12/dist-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.



📊 Resumen fase 1:
   Candidatos totales : 24
   Aprobados (pass)   : 7
   Top seleccionados  : 7

🏆 Top segmentos por score:
   • score=0.009  dur= 5.13s  snr=  4.6dB  2026-05-15 _Cita oftalmológica_0019_00408.02.wav
   • score=0.009  dur= 6.10s  snr=  3.8dB  2026-05-15 _Cita oftalmológica_0011_00217.98.wav
   • score=0.006  dur= 5.52s  snr=  3.7dB  2026-05-15 _Cita oftalmológica_0004_00119.76.wav
   • score=0.002  dur= 4.40s  snr=  4.2dB  2026-05-15 _Cita oftalmológica_0023_00645.37.wav
   • score=0.002  dur= 5.10s  snr=  3.4dB  2026-05-15 _Cita oftalmológica_0015_00292.11.wav
   • score=0.000  dur= 4.10s  snr=  3.7dB  2026-05-15 _Cita oftalmológica_0007_00167.98.wav
   • score=0.000  dur= 4.14s  snr=  3.3dB  2026-05-15 _Cita oftalmológica_0022_00486.59.wav
CPU times: user 1.8 s, sys: 424 ms, total: 2.23 s
Wall time: 3.01 s


In [8]:
import json
from pathlib import Path

# Leer el reporte más reciente
reportes = sorted(paths.reports.glob('reference_quality_*.json'), reverse=True)
reporte = json.loads(reportes[0].read_text(encoding='utf-8'))

print(f'Reporte: {reportes[0].name}')
print(f'Resumen: {reporte["summary"]}')
print()
print(f'{"#":<4} {"score":>6} {"dur(s)":>7} {"snr(dB)":>8} {"sr(Hz)":>7} {"pass":>5}  razón de fallo')
print('─' * 80)
for i, seg in enumerate(reporte['all_segments'], 1):
    print(
        f'{i:<4} {seg["score"]:>6.3f} {seg["duration_s"]:>7.2f} '
        f'{seg["snr_db"]:>8.1f} {seg["sample_rate"]:>7} '
        f'{"✅" if seg["passed"] else "❌":>5}  '
        f'{", ".join(seg["reasons"][:2]) if seg["reasons"] else ""}'
    )

Reporte: reference_quality_20260518T160954Z.json
Resumen: {'total_candidates': 24, 'passing': 7, 'top_selected': 7}

#     score  dur(s)  snr(dB)  sr(Hz)  pass  razón de fallo
────────────────────────────────────────────────────────────────────────────────
1     0.025    5.22      7.1   22050     ❌  clipping risk: peak 0.0dBFS
2     0.041    8.33      4.9   22050     ❌  clipping risk: peak 0.0dBFS
3     0.000    6.35      2.3   22050     ❌  snr 2.3dB < 3.0dB
4     0.000   11.58      3.0   22050     ❌  snr 3.0dB < 3.0dB, clipping risk: peak 0.0dBFS
5     0.005    5.52      3.7   22050     ✅  
6     0.003    6.10      3.3   22050     ❌  clipping risk: peak 0.0dBFS
7     0.000    6.10      1.8   22050     ❌  snr 1.8dB < 3.0dB
8     0.000    4.10      3.7   22050     ✅  
9     0.000    4.10      2.5   22050     ❌  snr 2.5dB < 3.0dB
10    0.000    7.10      2.3   22050     ❌  snr 2.3dB < 3.0dB
11    0.000    5.10      2.2   22050     ❌  snr 2.2dB < 3.0dB
12    0.009    6.10      3.8   22050

---

## 9️⃣ Escuchar las referencias top (control de calidad humano)

Antes de sintetizar, **escucha** los segmentos elegidos. Si alguno suena raro (eco, otra voz, palabras cortadas), elimínalo manualmente o ajusta los filtros y re-corre la fase 1.


In [9]:
from IPython.display import Audio, display
import json

# Mostrar metadata + reproductor por cada top WAV
for i, wav_path in enumerate(corpus_result.top_wavs, 1):
    print(f'\n[{i}/{len(corpus_result.top_wavs)}] {wav_path.name}')
    display(Audio(str(wav_path)))

# Imprimir ruta al reporte más reciente
reports = sorted(paths.reports.glob('reference_quality_*.json'), reverse=True)
if reports:
    print(f'\n📋 Reporte JSON detallado: {reports[0]}')
    with open(reports[0]) as f:
        summary = json.load(f).get('summary', {})
    print(f'   Resumen: {summary}')



[1/7] 2026-05-15 _Cita oftalmológica_0019_00408.02.wav



[2/7] 2026-05-15 _Cita oftalmológica_0011_00217.98.wav



[3/7] 2026-05-15 _Cita oftalmológica_0004_00119.76.wav



[4/7] 2026-05-15 _Cita oftalmológica_0023_00645.37.wav



[5/7] 2026-05-15 _Cita oftalmológica_0015_00292.11.wav



[6/7] 2026-05-15 _Cita oftalmológica_0007_00167.98.wav



[7/7] 2026-05-15 _Cita oftalmológica_0022_00486.59.wav



📋 Reporte JSON detallado: /content/drive/MyDrive/Pruebas/voicelegacy/reports/reference_quality_20260518T160954Z.json
   Resumen: {'total_candidates': 24, 'passing': 7, 'top_selected': 7}


---

## 🔟 Fase 2: Sintetizar los textos

Para cada texto en `TEXTS_TO_SYNTHESIZE`, genera un WAV con la voz clonada usando los top segmentos como condicionamiento.

Idempotente: si ya se sintetizó la misma combinación (texto + referencias + config), se reusa el archivo.


In [11]:
TEXTS_TO_SYNTHESIZE

['Mi querido nieto, hoy quiero contarte algo importante: la paciencia es la virtud que más vale la pena cultivar.',
 'Cuando tengas dudas, escucha a las personas que te aman, pero al final decide tú, con la cabeza fría.']

In [10]:
%%time
results = run_batch_synthesis(
    texts=TEXTS_TO_SYNTHESIZE,
    reference_wavs=corpus_result.top_wavs,
    paths=paths,
    config=config,
)

print()
print(f'📦 {len(results)} archivo(s) sintetizado(s):')
for r in results:
    marker = '♻️ ' if r.cached else '✨'
    print(f'  {marker} {r.output_path}')


16:10:43 | INFO     | voicelegacy.synthesis:load_xtts_model:78 | Loading XTTS model 'tts_models/multilingual/multi-dataset/xtts_v2' on device 'cuda'
16:11:01 | INFO     | voicelegacy.synthesis:load_xtts_model:84 | Model loaded and cached.
16:11:01 | INFO     | voicelegacy.pipeline:run_batch_synthesis:256 | [1/2] 'Mi querido nieto, hoy quiero contarte algo importante: la pa'...
16:11:01 | INFO     | voicelegacy.synthesis:load_xtts_model:67 | Using cached model: tts_models/multilingual/multi-dataset/xtts_v2|auto
16:11:01 | INFO     | voicelegacy.synthesis:synthesize_to_file:149 | Synthesizing 111 char(s) to mi-querido-nieto-hoy-quiero-contarte-algo-importante-la-paci_d6305022.wav with 7 ref file(s)
16:11:09 | INFO     | voicelegacy.synthesis:synthesize_to_file:171 | Wrote /content/drive/MyDrive/Pruebas/voicelegacy/synthesis_out/mi-querido-nieto-hoy-quiero-contarte-algo-importante-la-paci_d6305022.wav (24000 Hz)
16:11:09 | INFO     | voicelegacy.persistence:save_synthesis_record:158 | Per


📦 2 archivo(s) sintetizado(s):
  ✨ /content/drive/MyDrive/Pruebas/voicelegacy/synthesis_out/mi-querido-nieto-hoy-quiero-contarte-algo-importante-la-paci_d6305022.wav
  ✨ /content/drive/MyDrive/Pruebas/voicelegacy/synthesis_out/cuando-tengas-dudas-escucha-a-las-personas-que-te-aman-pero-_87d55c69.wav
CPU times: user 23.8 s, sys: 3.27 s, total: 27.1 s
Wall time: 31.7 s


---

## 1️⃣1️⃣ Escuchar resultados


In [12]:
from IPython.display import Audio, display

for r in results:
    print(f'\n📝 {r.text[:100]}{"..." if len(r.text) > 100 else ""}')
    print(f'   {r.output_path.name} (cached={r.cached})')
    display(Audio(str(r.output_path)))



📝 Mi querido nieto, hoy quiero contarte algo importante: la paciencia es la virtud que más vale la pen...
   mi-querido-nieto-hoy-quiero-contarte-algo-importante-la-paci_d6305022.wav (cached=False)



📝 Cuando tengas dudas, escucha a las personas que te aman, pero al final decide tú, con la cabeza fría...
   cuando-tengas-dudas-escucha-a-las-personas-que-te-aman-pero-_87d55c69.wav (cached=False)


---

## 1️⃣2️⃣ Empaquetar entregables (opcional)

Copia los WAVs sintetizados + reportes de calidad a una carpeta `entregables/` lista para compartir.


In [ ]:
import shutil
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
deliverables = paths.workspace / f'entregables_{ts}'
deliverables.mkdir(parents=True, exist_ok=True)

# Copiar audios
for r in results:
    if r.output_path.exists():
        shutil.copy2(r.output_path, deliverables / r.output_path.name)

# Copiar reporte más reciente
latest_report = sorted(paths.reports.glob('reference_quality_*.json'), reverse=True)
if latest_report:
    shutil.copy2(latest_report[0], deliverables / latest_report[0].name)

# Metadata del run
import json, sys
meta = {
    'timestamp': datetime.now().isoformat(),
    'voicelegacy_version': voicelegacy.__version__,
    'python_version': sys.version,
    'config': config.model_dump(mode='json'),
    'n_reference_segments': len(corpus_result.top_wavs),
    'n_synthesis_outputs': len(results),
    'texts': TEXTS_TO_SYNTHESIZE,
}
(deliverables / 'run_metadata.json').write_text(
    json.dumps(meta, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

print(f'📦 Entregables en: {deliverables}')
for p in sorted(deliverables.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'   • {p.name} ({size_kb:.1f} KB)')


---

## 1️⃣3️⃣ Liberar VRAM y limpiar estado


In [ ]:
import gc

release_model()
gc.collect()
print('🧹 VRAM liberada, estado limpio.')

# Para terminar la sesión de Colab (libera GPU del pool compartido):
# from google.colab import runtime
# runtime.unassign()


---

## 🔧 Troubleshooting

| Síntoma | Causa probable | Acción |
|---|---|---|
| `❌ No se detectó GPU` | Runtime sin GPU | Runtime → Change runtime type → T4 GPU |
| `RuntimeError: You must accept the Coqui Public Model License` | `ACCEPT_CPML=False` | Lee https://coqui.ai/cpml, cambia a `True` en celda 2 |
| `ModuleNotFoundError: No module named 'voicelegacy'` | Paquete no instalado | Re-corre la celda 4 |
| `Reference corpus is empty` | No hay JSONs / mal speaker / audio rechazado | Ver mensajes de la celda 8 |
| `CUDA out of memory` | Otro modelo cargado | Reinicia runtime |
| El clon suena "como en teléfono" | Audio fuente comprimido | Re-graba con micrófono decente; las llamadas no sirven |
| El clon ignora la entonación | Referencias muy cortas | Aumenta `MAX_SEGMENT_DURATION_S` a 20, baja `MIN_SEGMENT_DURATION_S` a 6 |
| Voice drift en textos largos | Texto > 250 caracteres sin pausas | `enable_text_splitting=True` (default); o divide el texto en oraciones cortas |

### Si quieres calidad superior a zero-shot

Necesitas **fine-tuning**, que requiere:
- ~1 hora de audio limpio + transcripciones en formato LJSpeech.
- 12–24 horas de GPU continua (RTX 4090 o equivalente).
- **No es viable en Colab Free**. Opciones: Colab Pro+ ($49.99/mes), RunPod ($0.34–0.79/h en RTX 4090), Lambda Labs.


---
## 🛑 Step 11 — Shut down runtime (releases GPU quota)

> ⚠️ **This cell will terminate your Colab session** if you uncomment the last line. All unsaved in-memory variables will be lost. Your Drive files are NOT affected.

Run this when you are completely done with the session to return the GPU to the Colab pool (important for staying within free-tier limits).

**Instructions**: uncomment the `runtime.unassign()` line and re-run the cell.

In [ ]:
# ⚠️  THIS CELL ENDS THE SESSION if you uncomment the last line.
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import gc, torch
import pytz

print(f"🕐 Hora Bogotá: {datetime.now(pytz.timezone('America/Bogota')).strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🕐 Current time (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}")

# Clean up remaining large variables
for var_name in list(globals().keys()):
    if var_name.startswith(("model", "results", "config", "paths")):
        try:
            del globals()[var_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n🛑 To shut down NOW, uncomment the line below and re-run this cell:")
print("    # from google.colab import runtime; runtime.unassign()")

# ─── UNCOMMENT THE LINE BELOW ONLY WHEN YOU ARE SURE ──────────────
from google.colab import runtime; runtime.unassign()

🕐 Hora Bogotá: 2026-05-17 22:52:39
🕐 Current time (UTC): 2026-05-18 03:52:39

🛑 To shut down NOW, uncomment the line below and re-run this cell:
    # from google.colab import runtime; runtime.unassign()
